Configuration

In [ ]:
model_name = "llava"
task_type = 'classification'
domain = 'Construction' # 'Traffic', 'Construction', 'Warehouse', 'Merged'
data_injection = f'{domain}More'
RULE_TEMPLATE = "coded-rules"
root_dir = "/content/drive/MyDrive/"
save_dir = f"{root_dir}/final_results_finetune"

# template_id = 't4'
# template = """
# Analyze the image against the rule set.

# {v}

# Respond with exactly one of: "Complied", "Violated", or "Not Applicable"."""

template_id = 'v1'
template = """
Analyze the image against the rule set.

{v}

Respond only with a JSON object containing a single key:
  - "classification": one of "Complied", "Violated", or "Not Applicable"."""

MAX_LENGTH = 128
ENTITY = 'szng-swinburne-university-of-technology'
PROJECT_NAME = f'{model_name}-{data_injection}-hazard-{task_type}-{template_id}'
REPO_ID = f"{model_name}-{data_injection}-hazard-{task_type}-{template_id}"


Install

In [ ]:
# @title
%%capture
# !pip install pandas
# !pip install scikit-learn
# !pip install wandb
# !pip install pillow
# !pip install torch torchvision torchaudio
# !pip install nltk
# !pip install peft
# !pip install -U datasets
# !pip install flash-attn --no-build-isolation
# !pip install transformers==4.49.0
!pip install lightning
!pip install json-repair
!pip install --upgrade transformers

Import

In [ ]:
# @title
%%capture
import requests
import torch
import time
import pandas as pd
from PIL import Image
from transformers import LlavaForConditionalGeneration, LlavaNextForConditionalGeneration, MllamaForConditionalGeneration, AutoProcessor
import os
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from collections import defaultdict
from tqdm.notebook import tqdm
import ast
from json_repair import repair_json

from google.colab import drive
drive.mount('/content/drive')

from IPython.display import HTML, display, clear_output
def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

import os

if not os.path.exists(save_dir):
    os.makedirs(save_dir)

Rule

In [ ]:
# @title
rule_description = {

"Construction": {

"Ladder Use" : """Ladder Use:
* Ladders shall be used only on stable and level surfaces unless secured to prevent accidental displacement. [1926.1053(b)(6)]
* The area around the top and bottom of ladders shall be kept clear. [1926.1053(b)(9)]
* When ascending or descending a ladder, the user shall face the ladder. [1926.1053(b)(20)]
* Each employee shall use at least one hand to grasp the ladder when progressing up and/or down the ladder. [1926.1053(b)(21)]
* An employee shall not carry any object or load that could cause the employee to lose balance and fall. [1926.1053(b)(22)]""",

"Protective Equipment": """Protective Equipment:
* Employees working in areas where there is a possible danger of head injury from impact, or from falling or flying objects, or from electrical shock and burns, shall be protected by protective helmets. [1926.100(a)]
* Each affected employee uses appropriate eye or face protection when exposed to eye or face hazards from flying particles, molten metal, liquid chemicals, acids or caustic liquids, chemical gases or vapors, or potentially injurious light radiation. [1926.102(a)(1)]
* Each employee on a walking/working surface (horizontal and vertical surface) with an unprotected side or edge which is 6 feet (1.8 m) or more above a lower level shall be protected from falling by the use of guardrail systems, safety net systems, or personal fall arrest systems. [1926.501(b)(1)]""",

"Fire Risk" : """Fire Safety:
* Smoking shall be prohibited at or in the vicinity of operations which constitute a fire hazard, and shall be conspicuously posted: "No Smoking or Open Flame." [1926.151(a)(3)]
* If the object to be welded, cut, or heated cannot be moved and if all the fire hazards cannot be removed, positive means shall be taken to confine the heat, sparks, and slag, and to protect the immovable fire hazards from them. [1926.352(b)]""",

"Crane Use" : """Crane Use:
* The operator must not engage in any practice or activity that diverts his/her attention while actually engaged in operating the equipment, such as the use of cellular phones (other than when used for signal communications). [1926.1417(d)]
* Erect and maintain control lines, warning lines, railings or similar barriers to mark the boundaries of the hazard areas. [1926.1424(a)(2)(ii)]
* While the operator is not moving a suspended load, no employee must be within the fall zone [1926.1425(b)]""",

"Scaffolding Risk" : """Scaffold Safety:
* Each platform on all working levels of scaffolds shall be fully planked or decked between the front uprights and the guardrail supports [1926.451(b)(1)]
* Guardrail systems shall be installed along all open sides and ends of platforms. [1926.451(g)(4)]
* The top edge height of toprails or equivalent member on supported scaffolds shall be installed between 38 inches (0.97 m) and 45 inches (1.2 m) above the platform surface. [1926.451(g)(4)(ii)]
* In addition to wearing hardhats each employee on a scaffold shall be provided with additional protection from falling hand tools, debris, and other small objects through the installation of toeboards, screens, or guardrail systems, or through the erection of debris nets, catch platforms, or canopy structures that contain or deflect the falling objects. [1926.451(h)(1)]""",
},

"Warehouse" : {

"Surface Condition" : """Surface Condition:
* All places of employment, passageways, storerooms, service rooms, and walking-working surfaces are kept in a clean, orderly, and sanitary condition. [29 CFR 1910.22(a)(1)]
* The floor of each workroom is maintained in a clean and, to the extent feasible, in a dry condition. When wet processes are used, drainage must be maintained and, to the extent feasible, dry standing places, such as false floors, platforms, and mats must be provided. [29 CFR 1910.22(a)(2)]
* Walking-working surfaces are maintained free of hazards such as sharp or protruding objects, loose boards, corrosion, leaks, spills, snow, and ice. [29 CFR 1910.22(a)(3)]""",

"Ergonomic Lifting" : """Ergonomic Lifting:
* Safe lifting involves— Holding the load close to your body at waist height. Never lift a heavy item above shoulder level. Never carry a load that obstructs your vision. [General Duty Clause, Section 5(a)(1)]
* The following points should be considered— The start and finish height of the load should be a suitable level above the floor, that is, between mid-thigh to shoulder height, preferably at about waist height. The back should not be twisted or bent sideways. Lifting with one hand should be avoided. [NOHSC:2005(1990) 5.66] """,

"Protective Equipment": """Protective Equipment:
* Each affected employee uses appropriate eye or face protection when exposed to eye or face hazards from flying particles, molten metal, liquid chemicals, acids or caustic liquids, chemical gases or vapors, or potentially injurious light radiation [29 CFR 1910.133(a)(1)]
* each affected employee wears a protective helmet when working in areas where there is a potential for injury to the head from falling objects. [29 CFR 1910.135(a)(1)]
* Personal fall protection systems must be worn with the attachment point of the body harness located in the center of the employee's back near shoulder level. The attachment point may be located in the pre-sternal position if the free fall distance is limited to 2 feet (0.6 m) or less. [29 CFR 1910.140(c)(22)]""",

"Ladder Use" : """Ladder Use:
* Ladders are used only on stable and level surfaces; [29 CFR 1910.23(c)(4)]
* Each employee faces the ladder when climbing up or down it; [29 CFR 1910.23(b)(11)]
* Each employee uses at least one hand to grasp the ladder when climbing up and down it; and [29 CFR 1910.23(b)(12)]
* No employee carries any object or load that could cause the employee to lose balance and fall while climbing up or down the ladder. [29 CFR 1910.23(b)(13)]""",

"Forklift Use" : """Forklift Use:
* No person shall be allowed to stand or pass under the elevated portion of any truck, whether loaded or empty. [29 CFR 1910.178(m)(2)]
* All traffic regulations shall be observed, including authorized plant speed limits. A safe distance shall be maintained approximately three truck lengths from the truck ahead, and the truck shall be kept under control at all times. [1910.178(n)(1)]
* The driver shall be required to look in the direction of, and keep a clear view of the path of travel. [1910.178(n)(6)]""",
},

"Traffic": {

"Driving Distraction" : """Driver Control:
* Distracted driving: Distracted driving is any activity that diverts attention from driving, including talking or texting on your phone, eating and drinking, talking to people in your vehicle, fiddling with the stereo, entertainment or navigation system — anything that takes your attention away from the task of safe driving. [National Highway Traffic Safety Administration]
* Driver to have proper control of a vehicle etc.: A person must not drive a vehicle if a person or an animal is in the driver's lap. [ROAD SAFETY ROAD RULES 2017 - REG 297 (1A)]
* Touching or looking at portable devices in motor vehicles: The driver of a motor vehicle must not touch a portable device in the motor vehicle while the vehicle is moving, or is stationary but not parked. [ROAD SAFETY ROAD RULES 2017 - REG 304J (1)]
* Duty of driver to avoid driving while fatigued: A person must not drive a fatigue-regulated heavy vehicle on a road while the person is impaired by fatigue. [HEAVY VEHICLE NATIONAL LAW (ACT) - SECT 228 (1)]""",

"Traffic Rules" : """Road Rules:
* Giving way at a pedestrian crossing: A driver must give way to any pedestrian on or entering a pedestrian crossing. [ROAD SAFETY ROAD RULES 2017 - REG 81 (2)]
* Overtaking or passing a vehicle at a children's crossing or pedestrian crossing: A driver approaching a children's crossing, or pedestrian crossing, must not overtake or pass a vehicle that is travelling in the same direction as the driver and is stopping, or has stopped, to give way to a pedestrian at the crossing. [ROAD SAFETY ROAD RULES 2017 - REG 82]
* Proceeding through a red traffic light: If traffic lights at an intersection or marked foot crossing are showing a red traffic light, a driver must not enter the intersection or marked foot crossing. [ROAD SAFETY ROAD RULES 2017 - REG 59]
* Driving on a one-way service road: A driver on the part of the road that is a service road must drive in the same direction as a vehicle travelling on the part of the road closest to the service road is required to travel. [ROAD SAFETY ROAD RULES 2017 - REG 136]
* Opening doors and getting out of a vehicle etc.: A person must not cause a hazard to any person or vehicle by opening a door of a vehicle, leaving a door of a vehicle open, or getting off, or out of, a vehicle. [ROAD SAFETY ROAD RULES 2017 - REG 269 (3)]
* Driving within a single marked lane or line of traffic: A driver on a multi-lane road must drive so the driver's vehicle is completely in a marked lane [ROAD SAFETY ROAD RULES 2017 - REG 146 (1)]
* Emergency stopping lane only signs: A driver must not drive in an emergency stopping lane. [ROAD SAFETY ROAD RULES 2017 - REG 95 (1)]
* Stopping in an emergency stopping lane: A driver must not stop in an emergency stopping lane. [ROAD SAFETY ROAD RULES 2017 - REG 178]
* Parking in parking bays: A driver must position the driver's vehicle completely within a single parking bay. [ROAD SAFETY ROAD RULES 2017 - REG 211 (2)]
* Obstructing access to and from a footpath, driveway etc.: A driver must not stop on a road in a position that obstructs access by vehicles or pedestrians to or from a footpath ramp or a similar way of access to a footpath, or a bicycle path or passageway. [ROAD SAFETY ROAD RULES 2017 - REG 198 (1)]""",

"Pedestrian Crossing" : """Pedestrian Rules:
* Crossing a road—general: A pedestrian crossing a road— (a) must cross by the shortest safe route; and (b) must not stay on the road longer than necessary to cross the road safely. [ROAD SAFETY ROAD RULES 2017 - REG 230 (1)]
* Crossing a road at pedestrian lights: If the pedestrian lights show a red pedestrian light and the pedestrian has not already started crossing the intersection or road, the pedestrian must not start to cross until the pedestrian lights change to green. [ROAD SAFETY ROAD RULES 2017 - REG 231 (2)]
* Pedestrians not to cause a traffic hazard or obstruction: A pedestrian must not cause a traffic hazard by moving into the path of a driver. [ROAD SAFETY ROAD RULES 2017 - REG 236 (1)]""",

"Road Condition" : """Driving Conditions:
* Obligations of road users: A person who drives a motor vehicle on a highway must drive in a safe manner having regard to all the relevant factors. [ROAD SAFETY ACT 1986 - SECT 17A (1)]
* The relevant factors include the following— (a) the physical characteristics of the road; (b) the prevailing weather conditions; (c) the level of visibility; (d) the condition of any vehicle the person is driving or riding on the highway; (e) the prevailing traffic conditions; (f) the relevant road laws and advisory signs; (g) the physical and mental condition of the driver or road user. [ROAD SAFETY ACT 1986 - SECT 17A (2A)]
* Relevant vehicle not to be used in hazardous area without hazardous area authority: A person must not use a relevant vehicle in a hazardous area. [ROAD SAFETY (VEHICLES) REGULATIONS 2021 - REG 299]""",

"Vehicle Load": """Vehicle Load:
* Carrying goods in addition to a large indivisible items: A load-carrying vehicle must not carry more than 1 large indivisible item. [HEAVY VEHICLE (MASS, DIMENSION AND LOADING) NATIONAL REGULATION - SCHEDULE 8 Division 2 - Load-carrying vehicles 13 (1)]
* Load restraint requirement: The following requirements apply to a vehicle that is carrying a load— (a) the load must be secured by a means that is appropriate to the vehicle and the nature of the load; (b) the load must be placed and secured on the vehicle in a way that prevents, or would be likely to prevent, the load or any part of the load from— (i) hanging or projecting from the vehicle; or (ii) becoming dislodged or falling from the vehicle; (c) the load must not be placed or secured on the vehicle in a way that makes the vehicle unstable; (d) the load must be placed and secured on the vehicle in compliance with the performance standards recommended in the Load Restraint Guide for Light Vehicles 2018, published by the National Transport Commission. [ROAD SAFETY (VEHICLES) REGULATIONS 2021 - REG 285]""",
}
}

custom_rules = rule_description #[domain]

# if domain == 'Merged':
#     custom_rules = {}
#     for d in ['Traffic', 'Construction', 'Warehouse']:
#         custom_rules.update(rule_description[d])


Prepare Data

In [ ]:
# @title
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

root_dir = '/content/drive/MyDrive/'

if domain == 'Merged':
    train_df = pd.DataFrame()
    val_df = pd.DataFrame()
    for d in ['Traffic', 'Construction', 'Warehouse']:
        df = pd.read_json(root_dir+f'experimentation/{d.lower()}-train.jsonl', lines=True)
        train_df = pd.concat([train_df, df])

        if 'More' in data_injection:
            trainmore_df = pd.read_json(root_dir + f'/experimentation/{domain.lower()}-train-more.jsonl', lines=True)
            trainmore_df = trainmore_df[trainmore_df['Image Path'] != 'data/processed_images/construction/Ladder_Use_1/0000066.jpg']
            train_df = pd.concat([train_df, trainmore_df]).reset_index()

        df = pd.read_json(root_dir+f'experimentation/{d.lower()}-val.jsonl', lines=True)
        val_df = pd.concat([val_df, df])
else:
    train_df = pd.read_json(root_dir+f'experimentation/{domain.lower()}-train.jsonl', lines=True)
    if 'More' in data_injection:
        trainmore_df = pd.read_json(root_dir + f'/experimentation/{domain.lower()}-train-more.jsonl', lines=True)
        trainmore_df = trainmore_df[trainmore_df['Image Path'] != 'data/processed_images/construction/Ladder_Use_1/0000066.jpg']
        train_df = pd.concat([train_df, trainmore_df]).reset_index()

    val_df = pd.read_json(root_dir+f'experimentation/{domain.lower()}-val.jsonl', lines=True)

train_df['Image Path'] = train_df['Image Path'].apply(lambda x: root_dir + x.replace('data/', ''))
val_df['Image Path'] = val_df['Image Path'].apply(lambda x: root_dir + x.replace('data/', ''))


In [ ]:
# @title

import re
import json

def parse_ruleset(raw_text, ruleset_id="RS-1"):
    lines = raw_text.strip().splitlines()
    ruleset_title = lines[0].rstrip(":").strip()
    rule_lines = [line.strip() for line in lines[1:] if line.strip()]

    rules = []
    rule_id_counter = 1

    for line in rule_lines:
        match = re.match(r"^\* (.*?):\s*(.*?)\s*\[(.*?)\]$", line)
        if match:
            title, definition, source = match.groups()
        else:
            source_match = re.search(r"\[(.*?)\]$", line)
            source = source_match.group(1) if source_match else "Unknown"
            definition = re.sub(r"^\*\s*", "", line)
            definition = re.sub(r"\s*\[.*?\]$", "", definition).strip()
            title = "General rule"

        rules.append({
            "ID": f"{ruleset_id}.{rule_id_counter}",
            "Title": title.strip(),
            "Definition": definition.strip(),
            "Source": source.strip()
        })
        rule_id_counter += 1

    ruleset = {
        "Rule Set": {
            "ID": ruleset_id,
            "Title": ruleset_title,
            "Rules": rules
        }
    }
    return ruleset

def json_to_ruleset(ruleset_json, indent_spaces = 2):
    ruleset = ruleset_json["Rule Set"]
    rule_lines = [f"Rule Set: {ruleset['Title']} (ID: {ruleset['ID']})"]
    indent = " " * indent_spaces

    for rule in ruleset["Rules"]:
        rule_lines.append(f"{indent}Rule ID: {rule['ID']}")
        rule_lines.append(f"{indent*indent_spaces}Title: {rule['Title']}")
        rule_lines.append(f"{indent*indent_spaces}Definition: {rule['Definition']}")
        rule_lines.append(f"{indent*indent_spaces}Source: {rule['Source']}")

    return "\n".join(rule_lines)

def get_cleaned_ruleset(rule_key, domain, rule_template = RULE_TEMPLATE):

    if rule_template == 'bulletpoints':

        if isinstance(rule_key, list):
            ruleset_text = "Rule Set: \n\n"
            for i, r in enumerate(rule_key, start=1):
                safety_rules = custom_rules[domain][r].strip()
                rule_name = safety_rules.split(':\n')[0].strip()
                rule_description = safety_rules[len(safety_rules.split(':\n')[0])+1:]
                ruleset_text += rule_description
                ruleset_text += "\n\n"

        elif rule_key == 'all':
            ruleset_text = "Rule Set: \n\n"
            for i, (k, v) in enumerate(custom_rules[domain].items(), start = 1):
                safety_rules = v.strip()
                rule_name = safety_rules.split(':\n')[0].strip()
                rule_description = safety_rules[len(safety_rules.split(':\n')[0])+1:]
                ruleset_text += rule_description
                ruleset_text += "\n\n"
        else:
            safety_rules = custom_rules[domain][rule_key].strip()
            rule_name = safety_rules.split(':\n')[0].strip()
            rule_description = safety_rules[len(safety_rules.split(':\n')[0])+1:]
            ruleset_text = f'Rule Set: {rule_description}"'

    elif rule_template == 'bulletpoints-with-header':

        if isinstance(rule_key, list):
          ruleset_text = "Rule Set: \n\n"
          for i, r in enumerate(rule_key, start=1):
              safety_rules = custom_rules[domain][r].strip()
              rule_name = safety_rules.split(':\n')[0].strip()
              rule_description = safety_rules[len(safety_rules.split(':\n')[0])+1:]
              ruleset_text += f'{i}. Rule: "{rule_name}" \nRelevant Sub-Rules: {rule_description}"'
              ruleset_text += "\n\n"

        elif rule_key == 'all':
            ruleset_text = "Rule Set: \n\n"
            for i, (k, v) in enumerate(custom_rules[domain].items(), start = 1):
                safety_rules = v.strip()
                rule_name = safety_rules.split(':\n')[0].strip()
                rule_description = safety_rules[len(safety_rules.split(':\n')[0])+1:]
                ruleset_text += f'{i}. Rule: "{rule_name}" \nRelevant Sub-Rules: {rule_description}"'
                ruleset_text += "\n\n"
        else:
            safety_rules = custom_rules[domain][rule_key].strip()
            rule_name = safety_rules.split(':\n')[0].strip()
            rule_description = safety_rules[len(safety_rules.split(':\n')[0])+1:]
            ruleset_text = f'Rule Set: "{rule_name}" \nRelevant Sub-Rules: {rule_description}"'

    elif rule_template == 'coded-rules':

        if isinstance(rule_key, list):
            ruleset_text = []
            for i, r in enumerate(rule_key, start=1):
                safety_rules = custom_rules[domain][r].strip()
                ruleset_json = parse_ruleset(safety_rules, ruleset_id=f"RS-{i}")
                ruleset_text.append(json_to_ruleset(ruleset_json))

            ruleset_text = '\n\n'.join(ruleset_text).strip()

        elif rule_key == 'all':
            ruleset_text = []
            for i, (k, v) in enumerate(custom_rules[domain].items(), start = 1):
                safety_rules = v.strip()
                ruleset_json = parse_ruleset(safety_rules, ruleset_id=f"RS-{i}")
                ruleset_text.append(json_to_ruleset(ruleset_json))

            ruleset_text = '\n\n'.join(ruleset_text).strip()

        else:
            safety_rules = custom_rules[domain][rule_key].strip()
            ruleset_json = parse_ruleset(safety_rules, ruleset_id="RS-1")
            ruleset_text = json_to_ruleset(ruleset_json).strip()

    return ruleset_text

def format_for_hf(df):

    formatted_data = []

    for _, row in df.iterrows():
        image = row['Image Path']

        if template_id.startswith('t'):
            ground_truth = row['Label']

        elif template_id.startswith('v'):
            ground_truth = f"""{{"classification": "{row['Label']}"}}"""

        elif template_id.startswith('r'):
            ground_truth = f"""{{"classification": "{row['Label']}", "explanation": "{row['Explanation']}"}}"""

        formatted_rules = get_cleaned_ruleset(row['Rule'], row['Domain'])

        prompt = template.format(v=formatted_rules)

        formatted_data.append({
            "id": row['File']+'_'+str(row['Image ID']),
            "image": image,
            "conversations": [
                {"role": "user",
                 "content": [
                     {"type": "image", "url": image},
                     {"type": "text", "text": prompt},
                  ]},
                {"role": "assistant",
                 "content": [{"type": "text", "text": ground_truth},]
                },
            ],
        })

    return formatted_data

import json

train_data = format_for_hf(train_df)
val_data = format_for_hf(val_df)

with open(f"train_llava_{task_type}.json", "w") as f:
    json.dump(train_data, f, indent=2)

with open(f"val_llava_{task_type}.json", "w") as f:
    json.dump(val_data, f, indent=2)

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

dataset = DatasetDict({'train': train_dataset, 'validation': val_dataset})
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'image', 'conversations'],
        num_rows: 3995
    })
    validation: Dataset({
        features: ['id', 'image', 'conversations'],
        num_rows: 500
    })
})

Model

In [ ]:
# @title
%%capture
from transformers import TrainingArguments, Trainer
from transformers.integrations import WandbCallback
from transformers import BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

if model_name == "llava":
    model_id = "llava-hf/llava-1.5-7b-hf"
    model = LlavaForConditionalGeneration.from_pretrained(model_id, device_map="auto", dtype=torch.float16)
    assistant_token = 'ASSISTANT:'

elif model_name == "llavanext":
    model_id = "llava-hf/llava-v1.6-mistral-7b-hf"
    model = LlavaNextForConditionalGeneration.from_pretrained(model_id, device_map="auto", dtype=torch.float16)
    assistant_token = '[/INST]'

processor = AutoProcessor.from_pretrained(model_id, use_fast=True)
processor.tokenizer.padding_side = "left"

Example

In [ ]:
# @title
conversation = train_dataset[1]['conversations']

prompt = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=False,
)
print(prompt[:-11])

USER: <image>

Analyze the image against the rule set.

Rule Set: Forklift Use (ID: RS-1)
  Rule ID: RS-1.1
    Title: General rule
    Definition: No person shall be allowed to stand or pass under the elevated portion of any truck, whether loaded or empty.
    Source: 29 CFR 1910.178(m)(2)
  Rule ID: RS-1.2
    Title: General rule
    Definition: All traffic regulations shall be observed, including authorized plant speed limits. A safe distance shall be maintained approximately three truck lengths from the truck ahead, and the truck shall be kept under control at all times.
    Source: 1910.178(n)(1)
  Rule ID: RS-1.3
    Title: General rule
    Definition: The driver shall be required to look in the direction of, and keep a clear view of the path of travel.
    Source: 1910.178(n)(6)

Respond only with a JSON object containing a single key:
  - "classification": one of "Complied", "Violated", or "Not Applicable". ASSISTANT: {"classification": "Not Applicable"}


LoRA Adapters

In [ ]:
# @title
def find_all_linear_names(model):
    cls = torch.nn.Linear
    lora_module_names = set()
    multimodal_keywords = ['mm_projector', 'vision_tower', 'vision_resampler']
    for name, module in model.named_modules():
        if any(mm_keyword in name for mm_keyword in multimodal_keywords):
            continue
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])

    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=find_all_linear_names(model),
    init_lora_weights="gaussian",
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# import re

# def find_lora_targets(model, top_mlp_layers=4):
#     multimodal_keywords = ['mm_projector', 'vision_tower', 'vision_resampler']
#     lora_module_names = []

#     linear_modules = [(name, module) for name, module in model.named_modules()
#                       if isinstance(module, torch.nn.Linear)
#                       and not any(mm in name for mm in multimodal_keywords)
#                       and 'lm_head' not in name]

#     # detect max layer number dynamically
#     layer_numbers = []
#     for name, module in linear_modules:
#         if 'mlp' in name:
#             match = re.search(r'\d+', name)
#             if match:
#                 layer_numbers.append(int(match.group()))
#     num_layers = max(layer_numbers) + 1 if layer_numbers else 0

#     for name, module in linear_modules:
#         if 'cross_attn' in name:
#             lora_module_names.append(name)
#         elif 'mlp' in name:
#             match = re.search(r'\d+', name)
#             if match:
#                 layer_num = int(match.group())
#                 if layer_num >= num_layers - top_mlp_layers:
#                     lora_module_names.append(name)

#     return lora_module_names


# target_modules = find_lora_targets(model, top_mlp_layers=4)

# lora_config = LoraConfig(
#     r=8,
#     lora_alpha=8,
#     lora_dropout=0.1,
#     target_modules=target_modules,
#     init_lora_weights="gaussian",
# )

# model = prepare_model_for_kbit_training(model)
# model = get_peft_model(model, lora_config)


LLava Train

In [ ]:
# @title
from torch.utils.data import Dataset
from typing import Any, Dict
import random
from datasets import load_dataset

class LlavaDataset(Dataset):
    """
    PyTorch Dataset for LLaVa. This class takes a HuggingFace Dataset as input.

    Each row, consists of image path(png/jpg/jpeg) and ground truth data (json/jsonl/txt).
    """

    def __init__(
        self,
        dataset_name_or_path: str,
        split: str = "train",
        sort_json_key: bool = True,
    ):
        super().__init__()

        self.split = split
        self.sort_json_key = sort_json_key

        self.dataset = load_dataset("json", data_files= dataset_name_or_path, split=self.split)
        self.dataset_length = len(self.dataset)

        self.gt_token_sequences = []
        for sample in self.dataset:

            self.gt_token_sequences.append([sample["conversations"]])

    def __len__(self) -> int:
        return self.dataset_length

    def __getitem__(self, idx: int) -> Dict:
        """
        Returns one item of the dataset.

        Returns:
            image : the original Receipt image
            target_sequence : tokenized ground truth sequence
        """
        sample = self.dataset[idx]

        image = Image.open(sample["image"])
        target_sequence = random.choice(self.gt_token_sequences[idx])  # can be more than one, e.g., DocVQA Task 1

        return image, target_sequence

train_dataset = LlavaDataset({"train": f"train_llava_{task_type}.json"},  split="train", sort_json_key=False)
val_dataset = LlavaDataset({"validation": f"val_llava_{task_type}.json"}, split="validation", sort_json_key=False)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [ ]:
# @title
def train_collate_fn(examples):
    images = []
    texts = []
    for example in examples:
        image, ground_truth = example
        images.append(image)
        input = processor.apply_chat_template(ground_truth, add_generation_prompt=True, tokenize=False,)
        texts.append(input[:-11])

    batch = processor(text=texts, images=images, padding=True, truncation=False, return_tensors="pt")

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels

    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]
    pixel_values = batch["pixel_values"]
    labels = batch["labels"]

    if model_name == 'llava':
      return input_ids, attention_mask, pixel_values, labels

    elif model_name == 'llavanext':
      image_sizes = batch["image_sizes"]
      return input_ids, attention_mask, pixel_values, image_sizes, labels

def eval_collate_fn(examples):
    images = []
    texts = []
    answers = []
    for example in examples:
        image, ground_truth = example
        images.append(image)
        input = processor.apply_chat_template(ground_truth[:-1], add_generation_prompt=True, tokenize=False,)
        texts.append(input)
        answers.append(ground_truth[-1]['content'][0]['text'])

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]
    pixel_values = batch["pixel_values"]

    if model_name == 'llava':
      return input_ids, attention_mask, pixel_values, answers

    elif model_name == 'llavanext':
      image_sizes = batch["image_sizes"]
      return input_ids, attention_mask, pixel_values, image_sizes, answers


In [ ]:
# @title

def flatten_dict(d):
    flat = {}

    def _flatten(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if isinstance(v, dict) or isinstance(v, list):
                    _flatten(v)
                else:
                    flat[k] = v
        elif isinstance(obj, list):
            for item in obj:
                _flatten(item)

    _flatten(d)
    return flat

def extract_json_prediction(raw_pred):
    try:
        prediction = flatten_dict(ast.literal_eval(repair_json(raw_pred)))

    except (ValueError, SyntaxError):
        return 'Unknown', '', -1

    classification = prediction.get('classification', prediction.get('initial_classification', 'Unknown'))
    try:
        reasoning = prediction.get('reasoning', ast.literal_eval(repair_json(raw_pred))['reasoning'])
    except:
        reasoning = ''
    explanation = prediction.get('explanation', '')
    confidence = float(prediction.get('confidence', -1))

    classification = sanitize_pred_label(classification)

    return classification, reasoning, explanation, confidence

def extract_xml_prediction(raw_pred):

    matching = re.search(r"<classification>\s*([^<]*?)\s*(?:</classification>|(?=<))", str(raw_pred), re.DOTALL)
    matching_ = re.search(r"<CONCLUSION>\s*([^<]*?)\s*(?:</CONCLUSION>|(?=<))", str(raw_pred), re.DOTALL)
    matching__ = re.search(r"<initial_classification>\s*([^<]*?)\s*(?:</initial_classification>|(?=<))", str(raw_pred), re.DOTALL)

    if matching:
        classification = matching.group(1).strip()
    elif matching_:
        classification = matching_.group(1).strip()
    elif matching__:
        classification = matching__.group(1).strip()
    else:
        classification = 'Unknown'

    matching2 = re.search(r"<reasoning>(.*?)</reasoning>", str(raw_pred), re.DOTALL)
    matching3 = re.search(r"<explanation>\s*([^<]*?)\s*(?:</explanation>|(?=<))", str(raw_pred), re.DOTALL)
    matching4 = re.search(r"<confidence>\s*([^<]*?)\s*(?:</confidence>|(?=<))", str(raw_pred), re.DOTALL)

    if matching2:
        reasoning = matching2.group(1).strip()
    else:
        reasoning = ''

    if matching3:
        explanation = matching3.group(1).strip()
    else:
        explanation = ''

    if matching4:
        try:
            confidence = float(matching4.group(1).strip())
        except:
            confidence = -1
    else:
        confidence = -1

    classification = sanitize_pred_label(classification)

    return classification, reasoning, explanation, confidence

def sanitize_pred_label(pred_label):

    pred_label = pred_label.replace('.', '')

    pred_label = pred_label.replace('Non-Compliant', 'Violated')
    pred_label = pred_label.replace('Compliant', 'Complied')
    pred_label = pred_label.replace('Non-Applicable', 'Not Applicable')

    pred_label = pred_label.strip().title()

    return pred_label if pred_label in ['Complied', 'Violated', 'Not Applicable'] else 'Unknown'

def get_model_outputs(img, template, max_new_tokens = 256):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": template}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=prompt, images=[img], return_tensors="pt").to(model.device)
    prediction = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        # repetition_penalty=1.2,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id
    )

    full_pred = processor.decode(prediction[0], skip_special_tokens=False)
    raw_pred = full_pred.split(assistant_token)[-1].replace(processor.tokenizer.eos_token, '').replace(processor.tokenizer.pad_token, '')

    return raw_pred, full_pred

def get_answers_from_raw_pred(template_id, raw_pred):

      if template_id.startswith('t') and model_name == 'llavacot':

          return extract_xml_prediction(raw_pred)

      if template_id.startswith('t'):

          return sanitize_pred_label(raw_pred), '', '', -1

      elif template_id.startswith('rx'):

          return extract_xml_prediction(raw_pred)

      elif template_id.startswith(('v', 'r')):

          return extract_json_prediction(raw_pred)



In [ ]:
# @title
import lightning as L
from torch.utils.data import DataLoader
import re
from nltk import edit_distance
import numpy as np
from sklearn.metrics import f1_score
import ast
from json_repair import repair_json
import torch
import torch.nn as nn
from collections import Counter
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_curve, roc_auc_score
from sklearn.preprocessing import MultiLabelBinarizer

class LlavaModelPLModule(L.LightningModule):
    def __init__(self, config, processor, model):
        super().__init__()
        self.config = config
        self.processor = processor
        self.model = model.to(torch.bfloat16)

        self.batch_size = config.get("batch_size")

    def training_step(self, batch, batch_idx):

        if model_name == 'llava':
            input_ids, attention_mask, pixel_values, labels = batch

            # no mask
            outputs = self.model(input_ids=input_ids,
                                    attention_mask=attention_mask,
                                    pixel_values=pixel_values,
                                    labels=labels)

        elif model_name == 'llavanext':
            input_ids, attention_mask, pixel_values, image_sizes, labels = batch

            # no mask
            outputs = self.model(input_ids=input_ids,
                                    attention_mask=attention_mask,
                                    pixel_values=pixel_values,
                                    image_sizes=image_sizes,
                                    labels=labels)

        # # mask

        # labels_masked = labels.clone()

        # gen_start_ids = self.processor.tokenizer(assistant_token, add_special_tokens=False).input_ids
        # gen_start_len = len(gen_start_ids)

        # for i in range(labels.size(0)):
        #     row_ids = input_ids[i].tolist()
        #     try:
        #         start_idx = row_ids.index(gen_start_ids[0])
        #     except ValueError:
        #         start_idx = 0

        #     labels_masked[i, :start_idx + gen_start_len] = -100

        # outputs = self.model(input_ids=input_ids,
        #                         attention_mask=attention_mask,
        #                         pixel_values=pixel_values,
        #                         labels=labels_masked)


        loss = outputs['loss']

        self.log("train_loss", loss, prog_bar=True)

        if batch_idx % 50 == 0:
            print(f"Step {self.global_step} | Training loss: {loss.item()}")

        return loss


    def validation_step(self, batch, batch_idx, dataset_idx=0):

        if model_name == 'llava':

            input_ids, attention_mask, pixel_values, labels = batch

            generated_ids = self.model.generate(input_ids=input_ids, attention_mask=attention_mask,
                                          pixel_values=pixel_values, max_new_tokens=MAX_LENGTH,
                                          pad_token_id=processor.tokenizer.pad_token_id,
                                          eos_token_id=processor.tokenizer.eos_token_id)
        elif model_name == 'llavanext':
            input_ids, attention_mask, pixel_values, image_sizes, labels = batch

            generated_ids = self.model.generate(input_ids=input_ids, attention_mask=attention_mask,
                                          pixel_values=pixel_values, max_new_tokens=MAX_LENGTH,
                                          image_sizes=image_sizes,
                                          pad_token_id=processor.tokenizer.pad_token_id,
                                          eos_token_id=processor.tokenizer.eos_token_id)

        predictions = self.processor.batch_decode(generated_ids[:, input_ids.size(1):], skip_special_tokens=True)

        # print(predictions)
        scores = []
        y_pred =[]
        y_true = []

        for pred, label in zip(predictions, labels):

            scores.append(edit_distance(pred, label) / max(len(pred), len(label)))

            if template_id.startswith(('v', 'r')):

                raw_pred = pred.replace(processor.tokenizer.eos_token, '').replace(processor.tokenizer.pad_token, '')

                pred, _, _, _ = get_answers_from_raw_pred(template_id, raw_pred)
                label, _, _, _ = get_answers_from_raw_pred(template_id, label)

            y_pred.append(pred)
            y_true.append(label)

        if "Unknown" in y_pred:
            print(y_pred, y_true)

        self.log("val_edit_distance", np.mean(scores), batch_size=self.batch_size,  prog_bar=True)

        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
        acc = accuracy_score(y_true, y_pred)

        # print('F1', f1)

        self.log("val_f1", f1, batch_size=self.batch_size, prog_bar=True)
        self.log("val_acc", acc, batch_size=self.batch_size, prog_bar=True)
        self.log("val_precision", precision, batch_size=self.batch_size, prog_bar=True)
        self.log("val_recall", recall, batch_size=self.batch_size, prog_bar=True)

        return scores

    def configure_optimizers(self):

        optimizer = torch.optim.AdamW(self.parameters(), lr=self.config.get("lr"))

        return optimizer

    def train_dataloader(self):
        return DataLoader(train_dataset, collate_fn=train_collate_fn, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(val_dataset, collate_fn=eval_collate_fn, batch_size=self.batch_size, shuffle=False)

config = {"max_epochs": 10,
          "val_check_interval": 0.2,
          "check_val_every_n_epoch": 1,
          "log_every_n_steps": 1,
          "gradient_clip_val": 1.0,
          "accumulate_grad_batches": 8,
          "lr": 1e-5,
          "batch_size": 1,
          "num_nodes": 1,
          "warmup_steps": 50,
          "result_path": "./result",
          "verbose": True,
          "seed": 42,
}

model_module = LlavaModelPLModule(config, processor, model)

from lightning.pytorch.callbacks import Callback
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from huggingface_hub import HfApi

api = HfApi()

class PushToHubCallback(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        print(f"Pushing model to the hub, epoch {trainer.current_epoch}")
        pl_module.model.push_to_hub(REPO_ID, private=True,
                                    commit_message=f"Training in progress, epoch {trainer.current_epoch}")

    def on_train_end(self, trainer, pl_module):
        print(f"Pushing model to the hub after training")
        pl_module.processor.push_to_hub(REPO_ID, private=True,
                                    commit_message=f"Training done")
        pl_module.model.push_to_hub(REPO_ID, private=True,
                                    commit_message=f"Training done")



from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

early_stop_callback = EarlyStopping(monitor="val_edit_distance", patience=3, verbose=True, mode="min")
checkpoint_callback = ModelCheckpoint(monitor='val_edit_distance', mode='min')

wandb_logger = WandbLogger(project=PROJECT_NAME, name=ENTITY, log_model='none')

trainer = L.Trainer(
        accelerator="gpu",
        devices=[0],
        max_epochs=config.get("max_epochs"),
        accumulate_grad_batches=config.get("accumulate_grad_batches"),
        check_val_every_n_epoch=config.get("check_val_every_n_epoch"),
        log_every_n_steps=config.get("log_every_n_steps"),
        gradient_clip_val=config.get("gradient_clip_val"),
        precision="bf16-mixed",
        num_sanity_val_steps=0,
        logger=wandb_logger,
        callbacks=[PushToHubCallback(), early_stop_callback],
)



INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [ ]:
# @title
trainer.fit(model_module)

INFO: You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO:lightning.pytorch.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
wandb: Currently logged in as: szng (szng-swinburne-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:231: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.
INFO: 
  | Name  | Type      | Params | Mode 
--------------------------------------------
0 | model | PeftModel | 7.1 B  | train
--------------------------------------------
21.3 M    Trainable params
7.1 B     Non-trainable params
7.1 B     Total params
28,338.807Total estimated model params size (MB)
2982      Modules in train mode
725       Modules in eval mode
INFO:lightning.pytorch.callbacks.model_summary:
  | Name  | Type      | Params | Mode 
--------------------------------------------
0 | model | PeftModel | 7.1 B  | train
--------------------------------------------
21.3 M    Trainable params
7.1 B     Non-trainab

Training: |          | 0/? [00:00<?, ?it/s]

Step 0 | Training loss: 10.334030151367188
Step 6 | Training loss: 9.658836364746094
Step 12 | Training loss: 9.116166114807129
Step 18 | Training loss: 7.526043891906738
Step 25 | Training loss: 6.363305568695068
Step 31 | Training loss: 5.959654808044434
Step 37 | Training loss: 4.967846393585205
Step 43 | Training loss: 4.180868625640869
Step 50 | Training loss: 3.7382099628448486
Step 56 | Training loss: 3.530093193054199
Step 62 | Training loss: 3.2682137489318848
Step 68 | Training loss: 3.481281042098999
Step 75 | Training loss: 2.9077258110046387
Step 81 | Training loss: 2.8643641471862793
Step 87 | Training loss: 2.803396701812744
Step 93 | Training loss: 3.0389204025268555
Step 100 | Training loss: 3.073864698410034
Step 106 | Training loss: 2.879441022872925
Step 112 | Training loss: 2.7154409885406494
Step 118 | Training loss: 2.6824233531951904
Step 125 | Training loss: 2.6821446418762207
Step 131 | Training loss: 2.7692627906799316
Step 137 | Training loss: 2.643146514892

Validation: |          | 0/? [00:00<?, ?it/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...cation-v1/tokenizer.model: 100%|##########|  500kB /  500kB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  98%|#########8| 41.9MB / 42.6MB            

Test

In [ ]:
%%capture
# @title
from transformers import LlavaForConditionalGeneration, LlavaNextForConditionalGeneration, MllamaForConditionalGeneration, AutoProcessor
import torch

model_id = 'Stephanienzz/'+REPO_ID
processor = AutoProcessor.from_pretrained(model_id, use_fast=True)
processor.tokenizer.padding_side = "left"

if model_name == "llava":
  my_model = LlavaForConditionalGeneration.from_pretrained(model_id, device_map="cuda:0", dtype=torch.float16, use_auth_token=True)
  assistant_token = 'ASSISTANT:'

elif model_name == "llavanext":
  my_model = LlavaNextForConditionalGeneration.from_pretrained(model_id, device_map="cuda:0", dtype=torch.float16, use_auth_token=True)
  assistant_token = '[/INST]'

processor = AutoProcessor.from_pretrained(model_id, use_fast=True)
processor.tokenizer.padding_side = "left"

In [ ]:
model_id

'Stephanienzz/llava-WarehouseMore-hazard-classification-v1'

In [ ]:
# @title

domain= 'Warehouse' # Traffic # Warehouse

root_dir = '/content/drive/MyDrive/'

test_df = pd.read_json(root_dir + f'experimentation/{domain.lower()}-test.jsonl', lines=True)

results = []

start_time = time.time()

for i, r in tqdm(test_df.iterrows(), total=len(test_df)):

    domain = r['Domain'].lower()
    filename = r['File']
    image_path = root_dir + r['Image Path'].replace('data/', '')
    image_id = r['Image ID']
    image_link = r['Image Link']
    rule_key = r['Rule']
    true_label = r['Label']

    # Defaults
    raw_pred = ''
    pred_label = 'Unknown'
    pred_reasoning = ''
    pred_explanation = ''
    pred_confidence = -1

    if not image_path.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    try:
        img = Image.open(image_path)
        if img.mode != 'RGB':
            img = img.convert('RGB')

        formatted_rules = get_cleaned_ruleset(rule_key, domain.title())
        classification_template = template.format(v=formatted_rules)

        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": classification_template}]}]
        prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

        inputs = processor(text=prompt, images=[img], return_tensors="pt").to(my_model.device)
        prediction = my_model.generate(**inputs, max_new_tokens=MAX_LENGTH, pad_token_id=processor.tokenizer.pad_token_id, eos_token_id=processor.tokenizer.eos_token_id)
        full_pred = processor.decode(prediction[0], skip_special_tokens=False)

        raw_pred = full_pred.split(assistant_token)[-1].replace(processor.tokenizer.eos_token, '').replace(processor.tokenizer.pad_token, '')

        pred_label, pred_reasoning, pred_explanation, pred_confidence = get_answers_from_raw_pred(template_id, raw_pred)

        results.append({
            'Domain': domain,
            'File': filename,
            'Image ID': image_id,
            'Image Path': image_path,
            'Image Link': image_link,
            'Rule': rule_key,
            'Label': true_label,
            'Pred Label': pred_label,
            'Pred Explanation': pred_explanation,
            'Pred Confidence': pred_confidence,
            'Pred Reasoning': pred_reasoning,
            'Pred Full Response': full_pred,
        })

            # print('\n', image_path, k, raw_pred)


    except Exception as e:
        print(f"\nPrediction failed for {image_path}: {e}")
        print(raw_pred)

        results.append({
            'Domain': domain,
            'File': filename,
            'Image ID': image_id,
            'Image Path': image_path,
            'Image Link': image_link,
            'Rule': rule_key,
            'Label': true_label,
            'Pred Label': pred_label,
            'Pred Explanation': pred_explanation,
            'Pred Confidence': pred_confidence,
            'Pred Reasoning': pred_reasoning,
            'Pred Full Response': full_pred,
        })

        continue

end_time = time.time()

total_time = end_time - start_time

mins, secs = divmod(total_time, 60)
print(f"Total execution time: {int(mins)} min {int(secs)} sec")

df_results = pd.DataFrame(results)
df_results.to_csv(f'{save_dir}/test_df-{domain}-{model_name}-{template_id}_finetuned_{data_injection}_{task_type}-coded-rules-{int(mins)}m{int(secs)}s.csv', index = False)

  0%|          | 0/500 [00:00<?, ?it/s]

Total execution time: 7 min 48 sec


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

print('F1 score: ', f1_score(df_results['Label'], df_results['Pred Label'], average="macro"))

F1 score:  0.7503203120693346
